# Background TOV

In [ ]:
c = 3e8  # speed of light in m/s
G = 6.67430e-11  # gravitational constant in m^3 kg^-1 s^-2
M_solar = 2e30 #kg 
scaling = (G/c**2)*10*M_solar

In [ ]:
import numpy as np
from scipy.integrate import solve_ivp
from scipy.optimize import root_scalar
from scipy.interpolate import InterpolatedUnivariateSpline, CubicSpline, interp1d
import matplotlib.pyplot as plt
import cmath as cm

# Constants
K = scaling #1.0             # Polytropic constant (dimensionless)
R_target = 1.0      # Target rescaled stellar radius
n = 1                # Polytropic index


"""def rho_from_p(p):
    p = np.maximum(p, 0.0)
    return pow((p / K),0.5) #np.sqrt(p / K)"""

def rho_from_p(p):
    return np.sqrt(np.maximum(p, 0.0) / K)

def energy_density_from_p(p):
    rho0 = rho_from_p(p)
    Gamma = 1 + 1/n
    return rho0 + (p/(Gamma-1))  # Total energy density for polytropic EOS


def tov_rhs(r, y):
    p, m, nu = y

    if p <= 0.0:
        return [0.0, 0.0, 0.0]

    rho0 = rho_from_p(p)
    eps  = energy_density_from_p(p) #rho0 + p  # total energy density

    dpdr = -(eps + p) * (m + 4*np.pi*r**3*p) / (r*(r - 2*m))
    dmdr = 4*np.pi * r**2 * eps
    dnudr = -2 * dpdr / (eps + p)

    return [dpdr, dmdr, dnudr]


# Solve TOV for a given p0
def solve_tov(p0, epsi_p = 1e-12): #changing to check for p_cut dependency
    y0 = [p0, 0.0,0.0]
    sol = solve_ivp(
        tov_rhs,
        [1e-6, 20],  # integrate far enough to catch where p=0
        y0,
        method='RK45',
        rtol=1e-10,
        atol=1e-12,
        dense_output=True
    )
    r_vals = sol.t
    p_vals = sol.y[0]
    m_vals = sol.y[1]
    
    p_cut = epsi_p * p0
    # Find where pressure becomes zero and interpolate to find R more accurately
    for i in range(1, len(p_vals)):
        if p_vals[i] <= p_cut:
            r1, r2 = r_vals[i-1], r_vals[i]
            p1, p2 = p_vals[i-1], p_vals[i]
            R_interp = r1 + (p_cut - p1) * (r2 - r1) / (p2 - p1)
            M_interp = m_vals[i-1] + (p_cut - p1) * (m_vals[i] - m_vals[i-1]) / (p2 - p1)
            return R_interp, M_interp, sol
    return r_vals[-1], m_vals[-1], sol  # fallback

# Shooting function to match compactness C = M/R
def shooting_target(p0, C_target):
    R, M, _ = solve_tov(p0)
    return M / R - C_target

# Define the values of compactness to solve for
C_vals = np.round(np.arange(0.10, 0.24, 0.01), 2)
profiles = {}


for C_target in C_vals:
    soln = root_scalar(
        shooting_target,
        args=(C_target,),
        bracket=[1e-6, 1.0],
        method='brentq',
        xtol=1e-12
    )

    if not soln.converged:
        raise RuntimeError(f"Root finding failed for C = {C_target}")

    p0 = soln.root
    R_raw, M_raw, sol = solve_tov(p0)
    scale_factor = R_target / R_raw
    print(R_target, R_raw, scale_factor)

    scale_factor = R_target / R_raw  # alpha

    r_vals = sol.t * scale_factor           # r -> alpha * r   (correct)
    p_vals = sol.y[0] * scale_factor**(-2)  # p -> alpha^{-2} * p    <-- FIXED
    m_vals = sol.y[1] * scale_factor        # m -> alpha * m         (correct)
    rho_vals = rho_from_p(p_vals)           # consistent with p
    nu_vals = sol.y[2]
    # Truncate where p goes to zero
    """
    mask = p_vals > p_cut
    r_vals = r_vals[mask]
    p_vals = p_vals[mask]
    m_vals = m_vals[mask]
    rho_vals = rho_vals[mask]
    nu_vals = nu_vals[mask]
    """
    
    mask = r_vals <= R_target
    r_vals = r_vals[mask]
    p_vals = p_vals[mask]
    m_vals = m_vals[mask]
    rho_vals = rho_vals[mask]
    nu_vals = nu_vals[mask]
    
    R = R_raw * scale_factor
    M = M_raw * scale_factor

    if 2.0 * M / R >= 1.0:
        raise ValueError(f"Unphysical compactness encountered for C = {C_target}: 2M/R >= 1")

    nu_target_surface = np.log(1.0 - 2.0 * M / R)
    nu_raw_surface = nu_vals[-1]

    # Compute constant shift Δν
    delta_nu = nu_target_surface - nu_raw_surface

    # Apply the shift to entire profile
    nu_vals_normalized = nu_vals + delta_nu


    profiles[C_target] = {
        'r': r_vals,
        'r_raw': R_raw,
        'p': p_vals,
        'm': m_vals,
        'rho': rho_vals,
        'TE': energy_density_from_p(p_vals),
        'nu': nu_vals,
        'nu_vals_norm': nu_vals_normalized,
        'p0': p0,
        'R': R_raw * scale_factor,
        'M': M_raw * scale_factor,
        'M_raw': M_raw
    }


for C in C_vals:
    plt.plot(profiles[C]['r'], profiles[C]['rho'], label=f'C = {C:.2f}')
plt.axvline(1.0, color='black', linestyle='--', label='r = 1')
plt.xlabel('r')
plt.ylabel('Density ρ(r)')
plt.title('Density profiles for various compactness values')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

# Pressure
#plt.figure(figsize=(10, 6))
#plt.subplot(3,1,2)
for C in C_vals:
    plt.plot(profiles[C]['r'], profiles[C]['p'], label=f'C = {C:.2f}')
plt.xlabel('Radius $r$')
plt.ylabel('Pressure $p(r)$')
plt.title('Pressure profiles for various $C$')
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

# Mass
#plt.figure(figsize=(10, 6))
#plt.subplot(3,1,3)
for C in C_vals:
    plt.plot(profiles[C]['r'], profiles[C]['m'], label=f'C = {C:.2f}')
plt.xlabel('Radius $r$')
plt.ylabel('Mass $m(r)$')
plt.title('Mass profiles for various $C$')
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

#Total energy
for C in C_vals:
    plt.plot(profiles[C]['r'], profiles[C]['TE'], label=f'C = {C:.2f}')
plt.axvline(1.0, color='black', linestyle='--', label='r = 1')
plt.xlabel('r')
plt.ylabel('Total Energy Density ')
plt.title('Total Energy profiles for various compactness values')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
r_vals = []
M_vals = []

for i in profiles:
    r_vals.append(profiles[i]['r_raw'])
    M_vals.append(profiles[i]['M_raw'])


c = 3e8  # speed of light in m/s
G = 6.67430e-11  # gravitational constant in m^3 kg^-1 s^-2
M_solar = 2e30 #kg 

scaling = (G/c**2)*10*M_solar

plt.plot(scaling*np.array(r_vals), scaling*np.array(M_vals), marker='o')

In [ ]:


Msun = 1.47664 # Solar mass in geometric units (km)
K = 100 * Msun**2
#K = 1.0
n = 1
Gamma = (1 + 1/n)  # Polytropic index

def rho_from_p(p):
    # ρ = ρ_b + p
    # ρ_b = (p/K)^(1/Γ)
    rho_b = (p / K)**(1.0 / Gamma)
    return rho_b + p

def cs2_from_p(p):
    # c_s^2 = dp/dρ
    rho_b = (p / K)**(1.0 / Gamma)
    return 1.0 / (rho_b / (Gamma * p) + 1.0)


def tov_rhs(r, y):
    m, p, nu = y
    
    if p <= 0:
        return [0.0, 0.0, 0.0]
    
    rho = rho_from_p(p)
    
    # dm/dr
    dmdr = 4.0 * np.pi * r**2 * rho
    
    # dp/dr
    dpdr = -(rho + p) * (m + 4.0 * np.pi * r**3 * p) / (r * (r - 2.0*m))
    
    # dν/dr
    dnudr = 2.0 * (m + 4.0 * np.pi * r**3 * p) / (r * (r - 2.0*m))
    
    return [dmdr, dpdr, dnudr]



def surface_event(r, y):
    return y[1]  # pressure

surface_event.terminal = True
surface_event.direction = -1

def compactness_from_pc(p_c):
    tov = solve_tov(p_c)
    return tov["M"] / tov["R"]

def solve_tov(p_central, r_max=50.0, r0=1e-6):
    
    # Central expansions
    rho_c = rho_from_p(p_central)
    
    m0 = (4.0/3.0) * np.pi * rho_c * r0**3
    p0 = p_central
    nu0 = 0.0  # temporary gauge choice
    
    y0 = [m0, p0, nu0]
    
    sol = solve_ivp(
        tov_rhs,
        (r0, r_max),
        y0,
        events=surface_event,
        rtol=1e-9,
        atol=1e-12,
        method="RK45"
    )
    
    if sol.t_events[0].size == 0:
        raise RuntimeError("Surface not found. Increase r_max.")
    
    R = sol.t_events[0][0]
    M = sol.y_events[0][0][0]
    nu_surface = sol.y_events[0][0][2]
    
    # Fix ν normalization so that
    # e^ν(R) = 1 - 2M/R
    nu_shift = np.log(1.0 - 2.0*M/R) - nu_surface
    
    nu_corrected = sol.y[2] + nu_shift
    
    # Build interpolating splines
    r_vals = sol.t
    m_vals = sol.y[0]
    p_vals = sol.y[1]
    
    m_spline = CubicSpline(r_vals, m_vals)
    p_spline = CubicSpline(r_vals, p_vals)
    nu_spline = CubicSpline(r_vals, nu_corrected)
    
    return {
        "M": M,
        "R": R,
        "r": r_vals,
        "m_spline": m_spline,
        "p_spline": p_spline,
        "nu_spline": nu_spline
    }

In [ ]:
tov = solve_tov(p_central=7.51e-5)

print("M =", tov["M"])
print("R =", tov["R"])
print("Compactness =", tov["M"]/tov["R"])

# New profile builder

In [ ]:
import numpy as np
import cmath as cm
from scipy.integrate import solve_ivp
from scipy.interpolate import CubicSpline
from scipy.optimize import brentq

pi = np.pi
Msun = 1.47664 # Solar mass in geometric units (km)
K = 100 * Msun**2
#K = 1.0
n = 1
Gamma = (1 + 1/n)  # Polytropic index


def rho_from_p(p, K):
    rho_b = (np.maximum(p, 1e-13) / K)**0.5
    return rho_b + p

def cs2_from_p(p, K):
    rho_b = (np.maximum(p, 1e-13) / K)**0.5
    return 1.0 / (rho_b/(2*p) + 1.0)



def tov_rhs(r, y, K):
    m, p, nu = y
    
    if p <= 0:
        return [0.0, 0.0, 0.0]
    
    rho = rho_from_p(p, K)
    
    dmdr = 4*pi*r**2*rho
    
    dpdr = -(rho + p)*(m + 4*pi*r**3*p)/(r*(r-2*m))
    
    dnudr = 2*(m + 4*pi*r**3*p)/(r*(r-2*m))
    
    return [dmdr, dpdr, dnudr]

def surface_event(r, y, K):
    return y[1]

surface_event.terminal = True
surface_event.direction = -1



def solve_tov(p_c, K, r0=1e-6, r_max=50.0):
    
    rho_c = rho_from_p(p_c, K)
    
    m0 = (4/3)*pi*rho_c*r0**3
    nu0 = 0.0
    
    y0 = [m0, p_c, nu0]
    
    sol = solve_ivp(
        lambda r, y: tov_rhs(r, y, K),
        (r0, r_max),
        y0,
        events=lambda r,y: surface_event(r,y,K),
        rtol=1e-9,
        atol=1e-12
    )
    
    if sol.t_events[0].size == 0:
        raise RuntimeError("Surface not found.")
    
    R = sol.t_events[0][0]
    M = sol.y_events[0][0][0]
    nu_surface = sol.y_events[0][0][2]
    
    mask = sol.t <= R
    r_vals = sol.t[mask]
    m_vals = sol.y[0][mask]
    p_vals = sol.y[1][mask]
    nu_vals = sol.y[2][mask]
    
    # Force last point to be exactly R
    r_vals[-1] = R
    m_vals[-1] = M
    p_vals[-1] = 0.0
    nu_vals[-1] = nu_surface
    
    # Normalize metric
    nu_shift = np.log(1 - 2*M/R) - nu_surface
    nu_vals = nu_vals + nu_shift
    
    return r_vals, m_vals, p_vals, nu_vals, M, R
"""
def find_pc_for_compactness(C_target, K):
    
    def f(p_c):
        r, m, p, nu, M, R = solve_tov(p_c, K)
        return M/R - C_target
    
    return brentq(f, 1e-4, 10.0)

"""
def find_pc_for_compactness(C_target, K):
    
    def f(p_c):
        r, m, p, nu, M, R = solve_tov(p_c, K)
        return M/R - C_target
    
    # Scan log-space for bracket
    p_scan = np.logspace(-6, 2, 50)
    
    f_vals = []
    for p in p_scan:
        try:
            f_vals.append(f(p))
        except:
            f_vals.append(np.nan)
    
    for i in range(len(p_scan)-1):
        if np.isnan(f_vals[i]) or np.isnan(f_vals[i+1]):
            continue
        if f_vals[i]*f_vals[i+1] < 0:
            return brentq(f, p_scan[i], p_scan[i+1])
    
    raise ValueError("Target compactness not bracketed.")

def build_background_for_C(C_target, K):
    
    p_c = find_pc_for_compactness(C_target, K)
    
    r, m, p, nu, M, R = solve_tov(p_c, K)
    
    eps = rho_from_p(p, K)
    cs2 = cs2_from_p(p, K)
    
    lam = np.log(1/(1 - 2*m/r))
    
    # Derivatives
    p_spline = CubicSpline(r, p)
    eps_spline = CubicSpline(r, eps)
    nu_spline = CubicSpline(r, nu)
    
    p_p = p_spline.derivative()(r)
    eps_p = eps_spline.derivative()(r)
    nu_p = nu_spline.derivative()(r)
    
    bg_dict = {
        C_target: {
            "M": M,
            "R": R,
            "r": r,
            "m": m,
            "p": p,
            "eps": eps,
            "nu": nu,
            "lam": lam,
            "nu_p": nu_p,
            "p_p": p_p,
            "eps_p": eps_p,
            "cs2": cs2
        }
    }
    
    return bg_dict



In [ ]:
def build_background_for_C(C_target, K):
    
    p_c = find_pc_for_compactness(C_target, K)
    
    r, m, p, nu, M, R = solve_tov(p_c, K)
    
    eps = rho_from_p(p, K)
    cs2 = cs2_from_p(p, K)
    
    lam = np.log(1/(1 - 2*m/r))
    
    # Smooth derivatives via splines
    p_spline = CubicSpline(r, p)
    eps_spline = CubicSpline(r, eps)
    nu_spline = CubicSpline(r, nu)
    
    p_p = p_spline.derivative()(r)
    eps_p = eps_spline.derivative()(r)
    nu_p = nu_spline.derivative()(r)
    
    return {
        "M": M,
        "R": R,
        "C": M/R,
        "r": r,
        "m": m,
        "p": p,
        "eps": eps,
        "nu": nu,
        "lam": lam,
        "nu_p": nu_p,
        "p_p": p_p,
        "eps_p": eps_p,
        "cs2": cs2
    }

def build_backgrounds_for_C_array(C_array, K):
    
    bg_dict = {}
    
    for C in C_array:
        print(f"Building star for C = {C:.6f}")
        bg_dict[C] = build_background_for_C(C, K)
    
    return bg_dict





In [ ]:
K = 100*(Msun)**2

#C_values = np.array([0.12, 0.14, 0.16])

C_values = np.round(np.arange(0.10, 0.24, 0.01), 2)

backgrounds = build_backgrounds_for_C_array(C_values, K)
print(" C        M        R")
print("-"*30)

for C in backgrounds:
    M = backgrounds[C]["M"]
    R = backgrounds[C]["R"]
    print(f"{C:.6f}  {M:.6f}  {R:.6f}")



M_vals = [backgrounds[C]["M"] for C in backgrounds]
R_vals = [backgrounds[C]["R"] for C in backgrounds]

plt.plot(R_vals, M_vals, 'o-')
plt.xlabel("R")
plt.ylabel("M")
plt.title("Mass-Radius curve")
plt.show()

In [ ]:
def scan_compactness(K):
    p_vals = np.logspace(-6, 2, 40)
    print(" p_c        C")
    print("--------------------")
    for p in p_vals:
        try:
            r, m, p_arr, nu, M, R = solve_tov(p, K)
            print(f"{p:10.4e}  {M/R:.6f}")
        except:
            print(f"{p:10.4e}  FAILED")

#scan_compactness(K)



def compute_compactness_curve(K, p_min=1e-6, p_max=1e3, npts=100):
    
    p_vals = np.logspace(np.log10(p_min), np.log10(p_max), npts)
    C_vals = []
    
    for p in p_vals:
        try:
            r, m, p_arr, nu, M, R = solve_tov(p, K)
            C_vals.append(M/R)
        except:
            C_vals.append(np.nan)
    
    return p_vals, np.array(C_vals)

# Compute
p_vals, C_vals = compute_compactness_curve(K)

# Plot
imax = np.nanargmax(C_vals)
print("Maximum C =", C_vals[imax])
print("At p_c =", p_vals[imax])

plt.figure(figsize=(6,4))
plt.plot(p_vals, C_vals)
plt.scatter(p_vals[imax], C_vals[imax], color='red')
plt.xscale("log")
plt.xlabel("Central Pressure $p_c$")
plt.ylabel("Compactness $C$")
plt.title("Compactness Curve (Maximum Marked)")
plt.grid(True)
plt.show()

def compute_MR_curve(K, p_min=1e-6, p_max=1e3, npts=100):
    
    p_vals = np.logspace(np.log10(p_min), np.log10(p_max), npts)
    M_vals = []
    R_vals = []
    
    for p in p_vals:
        try:
            r, m, p_arr, nu, M, R = solve_tov(p, K)
            M_vals.append(M)
            R_vals.append(R)
        except:
            M_vals.append(np.nan)
            R_vals.append(np.nan)
    
    return np.array(M_vals), np.array(R_vals)

M_vals, R_vals = compute_MR_curve(K)

plt.figure(figsize=(6,4))
plt.plot(R_vals, M_vals)
plt.xlabel("R")
plt.ylabel("M")
plt.title("Mass-Radius Curve")
plt.grid(True)
plt.show()

In [ ]:
K = 100*(1.47664)**2   # same as Mathematica

bg = backgrounds#build_background_for_C(0.1460474, K)

C = list(bg.keys())[0]
R = bg[C]["R"]
M = bg[C]["M"]

print("M =", M)
print("R =", R)
print("C =", M/R)

In [ ]:
list(backgrounds.keys())

In [ ]:
backgrounds[C].keys()

In [ ]:
backgrounds[0.2]['p']

In [ ]:

C = 0.2

M = backgrounds[C]['M']
R = backgrounds[C]['R']
nu_in = backgrounds[C]['nu']
r_in = backgrounds[C]['r']
m = backgrounds[C]['m']

def nu_out(r):
    return np.log(1 - 2*M/r)


exp_nu_in = np.exp(nu_in)
r_out = np.linspace(R, 5*R, 200)
exp_nu_out = np.exp(nu_out(r_out))


plt.figure(figsize=(7,5))
plt.plot(r_in, exp_nu_in, label=r'$e^{\nu(r)}_{\mathrm{inside}}$', color='tab:blue')
plt.plot(r_out, exp_nu_out, label=r'$e^{\nu(r)}_{\mathrm{outside}}$', color='tab:orange', linestyle='--')

plt.axvline(R, color='k', linestyle=':', label=r'$r=R$ (surface)')
plt.xlabel(r'$r$ [km]')
plt.ylabel(r'$e^{\nu(r)}$')
plt.title('Continuity of $e^{\\nu(r)}$ at the stellar surface')
plt.legend()
plt.grid(True, ls=':')
plt.show()


elam_inside = 1 - 2 * m / r_in

r_out = np.linspace(R, 5 * R, 200)
elam_outside = 1 - 2 * M / r_out

plt.figure(figsize=(6,5))
plt.plot(r_in, elam_inside, label=r"$e^{-\lambda(r)}_{\rm inside}$", lw=2)
plt.plot(r_out, elam_outside, '--', label=r"$e^{-\lambda(r)}_{\rm outside}$", lw=2)
plt.axvline(R, color='k', ls=':', label=r"$r = R$ (surface)")
plt.xlabel(r"$r$ [km]")
plt.ylabel(r"$e^{-\lambda(r)}$")
plt.title(r"Continuity of $e^{-\lambda(r)}$ at the stellar surface")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


In [ ]:
print("R from background:", R)
print("Last r_in:", r_in[-1])

In [ ]:
def make_bg_tables_from_profile(profile, r_grid):
    """
    profile = one entry from backgrounds[C]
    r_grid  = interpolation grid (np.linspace(eps, R, N))
    """

    r_prof = profile['r']
    
    bg_tab = {
        'r': r_grid,
        'm': np.interp(r_grid, r_prof, profile['m']),
        'p': np.interp(r_grid, r_prof, profile['p']),
        'eps': np.interp(r_grid, r_prof, profile['eps']),
        'nu': np.interp(r_grid, r_prof, profile['nu']),
        'lam': np.interp(r_grid, r_prof, profile['lam']),
        'nu_p': np.interp(r_grid, r_prof, profile['nu_p']),
        'p_prime': np.interp(r_grid, r_prof, profile['p_p']),
        'eps_prime': np.interp(r_grid, r_prof, profile['eps_p']),
        'cs2': np.interp(r_grid, r_prof, profile['cs2']),
    }

    # lam_p computed from definition instead of spline
    bg_tab['lam_p'] = (
        2*(bg_tab['m'] + 4*np.pi*r_grid**3*bg_tab['p'])
        / (r_grid*(r_grid - 2*bg_tab['m']))
    )

    return bg_tab

def pack_bg_quantities(bg_tab):
    return np.vstack([
        bg_tab['m'],
        bg_tab['p'],
        bg_tab['eps'],
        bg_tab['nu'],
        bg_tab['lam'],
        bg_tab['lam_p'],
        bg_tab['nu_p'],
        bg_tab['p_prime'],
        bg_tab['eps_prime'],
        bg_tab['cs2'],
    ])

profile = backgrounds[0.2]

R = profile['R']
r_start = 1e-6 
r_tab = np.linspace(r_start, R, 4000)

bg_tab = make_bg_tables_from_profile(profile, r_tab)
bg_packed = pack_bg_quantities(bg_tab)

In [ ]:
class ComplexSpline:
    def __init__(self, x, y, k=3):
        self.real_spline = CubicSpline(x, np.real(y))
        self.imag_spline = CubicSpline(x, np.imag(y))

    def __call__(self, x):
        return self.real_spline(x) + 1j * self.imag_spline(x)

    def derivative(self, n=1):
        der_real = self.real_spline.derivative(n)
        der_imag = self.imag_spline.derivative(n)
        return lambda x: der_real(x) + 1j * der_imag(x)


class SourceTensor:
  def S01(self, r, omega): return 0.0
  def S01p(self, r, omega): return 0.0
  def S00(self, r, omega): return 0.0
  def S0A(self, r, omega): return 0.0
  def SZ(self, r, omega): return 0.0
  def S0(self, r, omega): return 0.0
  def S0p(self, r, omega): return 0.0
  def S1(self, r, omega): return 0.0
  def S1p(self, r, omega): return 0.0
  def SOm(self, r, omega): return 0.0

def make_source(source, r_grid, omega):

    return {
        'S01':  np.asarray([source.S01(r,  omega) for r in r_grid]),
        'S01p': np.asarray([source.S01p(r, omega) for r in r_grid]),
        'S00':  np.asarray([source.S00(r,  omega) for r in r_grid]),
        'S0A':  np.asarray([source.S0A(r,  omega) for r in r_grid]),
        'SZ':   np.asarray([source.SZ(r,   omega) for r in r_grid]),
        'S0':   np.asarray([source.S0(r,   omega) for r in r_grid]),
        'S0p':  np.asarray([source.S0p(r,  omega) for r in r_grid]),
        'S1':   np.asarray([source.S1(r,   omega) for r in r_grid]),
        'S1p':  np.asarray([source.S1p(r,  omega) for r in r_grid]),
        'SOm':  np.asarray([source.SOm(r,  omega) for r in r_grid]),
    }

def pack_source(source_tables):
    return np.vstack([source_tables[k] for k in ('S01','S01p','S00','S0A','SZ','S0','S0p','S1','S1p','SOm')])



from numba import njit
"""
@njit()
def interp(r, r_grid, f_grid):
    i = np.searchsorted(r_grid, r) - 1
    if i < 0:
        i = 0
    if i >= r_grid.size - 1:
        i = r_grid.size - 2

    r0 = r_grid[i]
    r1 = r_grid[i+1]
    f0 = f_grid[i]
    f1 = f_grid[i+1]

    return f0 + (f1 - f0) * (r - r0) / (r1 - r0)
"""
@njit
def interp(x, x_grid, y_grid):

    n = x_grid.size

    # detect direction
    increasing = x_grid[0] < x_grid[-1]

    if increasing:
        i = np.searchsorted(x_grid, x) - 1
    else:
        # search on reversed logic
        i = np.searchsorted(x_grid[::-1], x) - 1
        i = n - 2 - i

    if i < 0:
        i = 0
    if i > n - 2:
        i = n - 2

    x0 = x_grid[i]
    x1 = x_grid[i+1]
    y0 = y_grid[i]
    y1 = y_grid[i+1]

    return y0 + (y1 - y0) * (x - x0) / (x1 - x0)
       
  
def init_conds_even_perfect(r, om, profile, sign=+1):

    p0 = profile['p'][0]
    eps0 = profile['eps'][0]
    nu0 = profile['nu'][0]

    Whinit = 1.0
    Kinit = sign * 1 #(eps0 + p0)

    pi = np.pi

    Xinit = (1/6)*np.exp(-nu0/2)*(p0+eps0)*(-3*om**2 *Whinit+ np.exp(nu0)*(3*Kinit + 8*pi*Whinit*(3*p0+eps0)))

    H1init = (2/3)*(Kinit + 4*pi*Whinit*(p0+eps0))

    return np.array([
        Xinit.real, Xinit.imag,
        Whinit.real, Whinit.imag,
        H1init.real, H1init.imag,
        Kinit.real, Kinit.imag
    ])

print(init_conds_even_perfect(r_tab[0], 0.5, bg_tab))


#Old init conds: [3.85586324e-14 0.00000000e+00 2.13262475e-19 0.00000000e+00 5.17056176e-37 0.00000000e+00 1.00000000e-12 0.00000000e+

In [ ]:
def init_conds_even_perfect(r, om, profile, mode):

    p0 = profile['p'][0]
    eps0 = profile['eps'][0]
    nu0 = profile['nu'][0]
    pi = np.pi

    if mode == 0:
        W0 = 1.0
        K0 = 0.0
    else:
        W0 = 0.0
        K0 = 1.0

    X0 = (1/6)*np.exp(-nu0/2)*(p0+eps0)*(
            -3*om**2 *W0
            + np.exp(nu0)*(3*K0 + 8*pi*W0*(3*p0+eps0))
         )

    H10 = (2/3)*(K0 + 4*pi*W0)*(p0+eps0)

    return np.array([
        X0.real, X0.imag,
        W0, 0.0,
        H10.real, H10.imag,
        K0, 0.0
    ])

In [ ]:

@njit
def even_rhs_two_basis(x, y, x_grid, r_grid, bg_tab, src_tab, om, l=2):

    dydr = np.zeros_like(y)
    r = interp(x, x_grid, r_grid)
    print("Interpolated r:", r, "for x =", x, "with grid from", r_grid[0], "to", r_grid[-1], "x_grid:", x_grid[0], "to", x_grid[-1])
    # unpack background
    m   = interp(r, r_grid, bg_tab[0])
    p   = interp(r, r_grid, bg_tab[1])
    eps = interp(r, r_grid, bg_tab[2])
    nu  = interp(r, r_grid, bg_tab[3])
    lam = interp(r, r_grid, bg_tab[4])
    lam_p = interp(r, r_grid, bg_tab[5])
    nu_p  = interp(r, r_grid, bg_tab[6])
    p_prime   = interp(r, r_grid, bg_tab[7])
    eps_prime = interp(r, r_grid, bg_tab[8])
    cs2   = interp(r, r_grid, bg_tab[9])

    # unpack sources (all zero for perfect fluid)
    S01  = interp(r, r_grid, src_tab[0])
    S01p = interp(r, r_grid, src_tab[1])
    S00  = interp(r, r_grid, src_tab[2])
    S0A  = interp(r, r_grid, src_tab[3])
    SZ   = interp(r, r_grid, src_tab[4])
    S0   = interp(r, r_grid, src_tab[5])
    S0p  = interp(r, r_grid, src_tab[6])
    S1   = interp(r, r_grid, src_tab[7])
    S1p  = interp(r, r_grid, src_tab[8])
    SOm  = interp(r, r_grid, src_tab[9])


    exp_lam = np.exp(lam)
    exp_nu  = np.exp(nu)

    for offset in (0, 8):  # loop over two bases

        Xr = y[offset+0]
        Xi = y[offset+1]
        Wr = y[offset+2]
        Wi = y[offset+3]
        H1r = y[offset+4]
        H1i = y[offset+5]
        Kr  = y[offset+6]
        Ki  = y[offset+7]

        X = Xr + 1j*Xi
        W = Wr + 1j*Wi
        H1 = H1r + 1j*H1i
        K = Kr + 1j*Ki

        # ---- algebraic H0 and V here ----
        # (use your already corrected expressions)

        #Sighnum = cm.exp(-lam + (nu/2))*(2*cm.exp(lam)*r*om*S00*rho + p *(2*cm.exp(lam)*r*om*S00 + 1j*(12*cm.exp(lam)*S0A - 2*r* S01p + S01*(-4+r*lam_p - r*nu_p))) + 1j*(2*rho*(6*cm.exp(lam)*S0A - r*S01p) + S01*(2*r*n0*T*sp + rho*(-4+r*lam_p - r*nu_p))))
        Sigh = 0 #Sighnum/(2*r**3 *om*n0*T*(rho+p))
        drhods = 0

        Eh = (X/cs2) + drhods*Sigh


        Hdenom = r*(2*r + 3*m + 4*pi*r**3 *p) 
        H = (8*pi*r**4 * X * np.exp(-nu/2) + 2*r**2 *K - np.exp(-nu)*r**4 *om**2 *K + np.exp(lam)*r*K*m - 3*np.exp(lam)*K*m**2 + 4*np.exp(lam)*pi*r**4 *p*K - 16*np.exp(lam)*pi*r**3 *K *m*p - 16*np.exp(lam)*pi**2 *r**6 *K *p**2 + H1*(-3*r*m + r**4 *(np.exp(-lam-nu)*om**2 - 12*pi*p)) - 8*pi*r**2 *S0 - 16*np.exp(-lam)*pi*r**2 *S1)/Hdenom 

        Vhnum1 = np.exp(-lam + (nu/2))*(np.exp(lam) * r**2 * H *(p+eps) + 2*(-2*np.exp(lam)*SZ + np.exp(lam)*SOm + S1*(2+np.exp(lam)+4*np.exp(lam)*r**2 *(p-eps)) - np.exp(lam/2)*r*W*p_prime + r*S1p))
        Vhnum2 = np.exp(nu/2) *(2*X - (Vhnum1/r**2))
        Vh = Vhnum2/(2*om**2 *(eps+p))

        dH1dr = (np.exp(lam)/r) * H + (4*pi*r*(eps-p)*np.exp(lam) - (2*m*np.exp(lam)/r**2) - (3/r))*H1 + (np.exp(lam)/r)*K - 16*pi*((eps+p)/r)*np.exp(lam)*Vh + 16*pi*(np.exp(lam)/r) * SZ
        dKdr = (H/r) + (l*(l+1)/2*r) *H1 + ((nu_p/2) - ((l+1)/r))*K - 8*pi*((eps+p)/r)*np.exp(lam/2)*W + (16*pi/r)*SZ
        dWhdr = (r/2)*np.exp(lam/2)*H + r*np.exp(lam/2)*K + (r*np.exp((lam-nu)/2)/(eps+p)) * Eh - W*(l+1)/r -(l*(l+1)/r)*np.exp(lam/2)*Vh - np.exp(lam/2)*S00/(r*(eps+p)) + 8*pi*r*np.exp(lam/2)*SZ
        dXhdr = ((eps+p)/2)*np.exp(nu/2)*((1/r)-(nu_p/2))*H + ((eps+p)/2)*np.exp(nu/2)*(om**2 *r*np.exp(-nu) + (l*(l+1)/2*r))*H1 + ((eps+p)/2)*np.exp(nu/2)*((3*nu_p/2)-(1/r))*K - (l/r)*X - ((eps+p)/r)*np.exp((lam+nu)/2)*(om**2 *np.exp(-nu) + (np.exp(-lam)/4*r**2) *(np.exp(2*lam)*(1+8*pi*r**2 *p)**2 + 2*np.exp(lam)*(3 + 8*pi*r**2 *p - 7)) )*W + (l*(l+1)/r**2)*np.exp(nu/2)*p_prime*Vh + (np.exp(nu/2)/(2*r**3))*(2*l-1 +np.exp(lam)*(1+8*pi*r**2 *p))*S0 + (np.exp(nu/2)/r**2)*S0p - (np.exp(nu/2)/r**3)*(l*(l+1) + 8*pi*r**2 *(eps+p))*S1 - (2*np.exp(nu/2)/r**3)*SOm
        drdx = p / p_prime

        dH1dx = drdx * dH1dr
        dKdx  = drdx * dKdr
        dWhdx = drdx * dWhdr
        dXdx  = drdx * dXhdr
        # store real/imag parts
        dydr[offset+0] = np.real(dXdx)
        dydr[offset+1] = np.imag(dXdx)
        dydr[offset+2] = np.real(dWhdx)
        dydr[offset+3] = np.imag(dWhdx)
        dydr[offset+4] = np.real(dH1dx)
        dydr[offset+5] = np.imag(dH1dx)
        dydr[offset+6] = np.real(dKdx)
        dydr[offset+7] = np.imag(dKdx)

    return dydr

In [ ]:
x = ln(p)

In [ ]:
#init_conds_even_perfect(r_tab[0], omega, profile, +1)

In [ ]:

def EvenSolver_two_basis(omega, profile, bg_packed, r_tab, src_packed):
    start = 5

    """
    y0A = init_conds_even_perfect(r_tab[start], omega, profile, 0)
    y0B = init_conds_even_perfect(r_tab[start], omega, profile, 1)
    y0 = np.concatenate([y0A, y0B])
    """
    y0A = init_conds_even_perfect(r_tab[0], omega, profile, +1)
    y0B = init_conds_even_perfect(r_tab[0], omega, profile, -1)
    y0 = np.concatenate([y0A, y0B])
    
    p_tab = bg_packed[1].copy()
    r_tab = r_tab.copy()

    # remove p=0 surface
    mask = p_tab > 0.0

    p_tab = p_tab[mask]
    r_tab = r_tab[mask]
    bg_packed = bg_packed[:, mask]
    src_packed = src_packed[:, mask]
    print("len(p_tab) =", len(p_tab))
    x_tab = np.log(p_tab)

    x_tab = np.ascontiguousarray(x_tab)
    r_tab = np.ascontiguousarray(r_tab)
    bg_packed = np.ascontiguousarray(bg_packed)
    src_packed = np.ascontiguousarray(src_packed)

    x0 = x_tab[start]      # center (largest)
    xf = x_tab[-1]     # surface (smallest)
    print("x0 =", x0)
    print("xf =", xf)
    print("difference =", xf - x0)
    sol = solve_ivp(
        lambda x, y: even_rhs_two_basis(
            x, y,
            x_tab,
            r_tab,
            bg_packed,
            src_packed,
            omega
        ),
        (x0, xf),      # decreasing interval is fine
        y0,
        method='Radau',
        rtol=1e-8,
        atol=1e-10, 
        max_step=1e-3
       
    )

    yR = sol.y[:, -1]
    print("sol.t.shape:", sol.t.shape)
    # normalize two bases separately
    yR[0:8]  /= np.max(np.abs(yR[0:8]))
    yR[8:16] /= np.max(np.abs(yR[8:16]))

    XA = yR[0] + 1j*yR[1]
    XB = yR[8] + 1j*yR[9]
    print("XA =", XA, "XB =", XB)
    Cmatch = -XA/XB

    H1R = (yR[4] + 1j*yR[5]) + Cmatch*(yR[12] + 1j*yR[13])
    KR  = (yR[6] + 1j*yR[7]) + Cmatch*(yR[14] + 1j*yR[15])
    print("H1RA =", (yR[4] + 1j*yR[5]), "H1RB =", (yR[12] + 1j*yR[13]), "Cmatch =", Cmatch, "H1R =", H1R)
    print("KRA =", (yR[6] + 1j*yR[7]), "KRB =", (yR[14] + 1j*yR[15]), "KR =", KR)
    return H1R, KR, Cmatch

In [ ]:
1e-16

In [ ]:
source_dict = make_source(SourceTensor(), r_tab, 0.5)

bg_packed = pack_bg_quantities(bg_tab)
src_packed = pack_source(source_dict)

EvenSolver_two_basis(0.5, profile, bg_packed, r_tab, src_packed)

In [ ]:
H1R = H1A - H1A 

In [ ]:
def Zerilli_from_H1K(omega, R, M, H1R, KR, l=2):

    lam = (l - 1) * (l + 2)
    f = 1 - (2*M)/R

    D = lam * R + 6.0 * M

    # Zerilli function at surface
    ZR = R**3 * ((2*M - R)*H1R + R*KR) / (3*M + 2*R)

    # dZ/dr at surface
    dZR = (
        R**2 * (
            6*(2*M**3 + M**2*R + M*R**2 - R**3)*H1R
            + R*(-3*M**2 - 6*M*R + 2*R**2)*KR
        )
    ) / ((2*M - R)*(3*M + 2*R)**2)

    return ZR, dZR

In [ ]:
H1R, KR, Cmatch = EvenSolver_two_basis(0.5, profile, bg_packed, r_tab, src_packed)
ZR, dZR = Zerilli_from_H1K(0.5, R, M, H1R, KR)

print("ZR =", ZR, "dZR =", dZR, "dZR/ZR =", ((2*M)/(1-2*(M/R))) * dZR/ZR)

In [ ]:
@njit
def evenSystem_zoom(r, y, om, r_grid, bg_tab, source, l=2):
    Xhre, Xhim, Whre, Whim, H1re, H1im, Kre, Kim = y
    pi = cm.pi
    X = Xhre + 1j*Xhim
    W = Whre + 1j*Whim
    H1 = H1re + 1j*H1im
    K = Kre + 1j*Kim

    m = interp(r, r_grid, bg_tab[0])
    p = interp(r, r_grid, bg_tab[1])
    eps = interp(r, r_grid, bg_tab[10]) #interp(r, r_grid, bg_tab[2])
    nu = interp(r, r_grid, bg_tab[3])
    lam = interp(r, r_grid, bg_tab[4])
    lam_p = interp(r, r_grid, bg_tab[5])
    nu_p = interp(r, r_grid, bg_tab[6])
    p_prime = interp(r, r_grid, bg_tab[7])
    eps_prime = interp(r, r_grid, bg_tab[11])
    cs2 = interp(r, r_grid, bg_tab[9])
    
    if np.all(source == 0):
        S01 = 0.0
        S01p = 0.0
        S00 = 0.0
        S0A = 0.0
        SZ = 0.0
        S0 = 0.0
        S0p = 0.0
        S1 = 0.0
        S1p = 0.0
        SOm = 0.0
    else:
        S01  = interp(r, r_grid, source[0])
        S01p = interp(r, r_grid, source[1])
        S00  = interp(r, r_grid, source[2])
        S0A  = interp(r, r_grid, source[3])
        SZ   = interp(r, r_grid, source[4])
        S0   = interp(r, r_grid, source[5])
        S0p  = interp(r, r_grid, source[6])
        S1   = interp(r, r_grid, source[7])
        S1p  = interp(r, r_grid, source[8])
        SOm  = interp(r, r_grid, source[9])

    
    #Sighnum = cm.exp(-lam + (nu/2))*(2*cm.exp(lam)*r*om*S00*rho + p *(2*cm.exp(lam)*r*om*S00 + 1j*(12*cm.exp(lam)*S0A - 2*r* S01p + S01*(-4+r*lam_p - r*nu_p))) + 1j*(2*rho*(6*cm.exp(lam)*S0A - r*S01p) + S01*(2*r*n0*T*sp + rho*(-4+r*lam_p - r*nu_p))))
        Sigh = 0 #Sighnum/(2*r**3 *om*n0*T*(rho+p))
        drhods = 0

        Eh = (X/cs2) + drhods*Sigh


        Hdenom = r*(2*r + 3*m + 4*pi*r**3 *p) 
        H = (8*pi*r**4 * X * np.exp(-nu/2) + 2*r**2 *K - np.exp(-nu)*r**4 *om**2 *K + np.exp(lam)*r*K*m - 3*np.exp(lam)*K*m**2 + 4*np.exp(lam)*pi*r**4 *p*K - 16*np.exp(lam)*pi*r**3 *K *m*p - 16*np.exp(lam)*pi**2 *r**6 *K *p**2 + H1*(-3*r*m + r**4 *(np.exp(-lam-nu)*om**2 - 12*pi*p)) - 8*pi*r**2 *S0 - 16*np.exp(-lam)*pi*r**2 *S1)/Hdenom 

        Vhnum1 = np.exp(-lam + (nu/2))*(np.exp(lam) * r**2 * H *(p+eps) + 2*(-2*np.exp(lam)*SZ + np.exp(lam)*SOm + S1*(2+np.exp(lam)+4*np.exp(lam)*r**2 *(p-eps)) - np.exp(lam/2)*r*W*p_prime + r*S1p))
        Vhnum2 = np.exp(nu/2) *(2*X - (Vhnum1/r**2))
        Vh = Vhnum2/(2*om**2 *(eps+p))

        dH1dr = (np.exp(lam)/r) * H + (4*pi*r*(eps-p)*np.exp(lam) - (2*m*np.exp(lam)/r**2) - (3/r))*H1 + (np.exp(lam)/r)*K - 16*pi*((eps+p)/r)*np.exp(lam)*Vh + 16*pi*(np.exp(lam)/r) * SZ
        dKdr = (H/r) + (l*(l+1)/2*r) *H1 + ((nu_p/2) - ((l+1)/r))*K - 8*pi*((eps+p)/r)*np.exp(lam/2)*W + (16*pi/r)*SZ
        dWhdr = (r/2)*np.exp(lam/2)*H + r*np.exp(lam/2)*K + (r*np.exp((lam-nu)/2)/(eps+p)) * Eh - W*(l+1)/r -(l*(l+1)/r)*np.exp(lam/2)*Vh - np.exp(lam/2)*S00/(r*(eps+p)) + 8*pi*r*np.exp(lam/2)*SZ
        dXhdr = ((eps+p)/2)*np.exp(nu/2)*((1/r)-(nu_p/2))*H + ((eps+p)/2)*np.exp(nu/2)*(om**2 *r*np.exp(-nu) + (l*(l+1)/2*r))*H1 + ((eps+p)/2)*np.exp(nu/2)*((3*nu_p/2)-(1/r))*K - (l/r)*X - ((eps+p)/r)*np.exp((lam+nu)/2)*(om**2 *np.exp(-nu) + (np.exp(-lam)/4*r**2) *(np.exp(2*lam)*(1+8*pi*r**2 *p)**2 + 2*np.exp(lam)*(3 + 8*pi*r**2 *p - 7)) )*W + (l*(l+1)/r**2)*np.exp(nu/2)*p_prime*Vh + (np.exp(nu/2)/(2*r**3))*(2*l-1 +np.exp(lam)*(1+8*pi*r**2 *p))*S0 + (np.exp(nu/2)/r**2)*S0p - (np.exp(nu/2)/r**3)*(l*(l+1) + 8*pi*r**2 *(eps+p))*S1 - (2*np.exp(nu/2)/r**3)*SOm


    return np.array([dXhdr.real, dXhdr.imag, dWhdr.real, dWhdr.imag, dH1dr.real, dH1dr.imag, dKdr.real, dKdr.imag])


def even_wrapper(r, y, om, r_grid, bg_tab, source):
    return evenSystem_zoom(r, y, om, r_grid, bg_tab, source, l=2)


def EvenSolver_single_basis(omega, profile, bg_packed, r_tab):

    R = profile['R']

    # small offset from center to avoid 1/r singularity
    r0 = r_tab[5]

    # initial conditions (single choice)
    y0 = init_conds_even_perfect(r0, omega, profile, mode=1)

    sol = solve_ivp(
        lambda r, y: evenSystem_zoom(r, y, omega, r_tab, bg_packed, source=np.zeros((10, r_tab.size)), l=2),
        (r0, R),
        y0,
        method='DOP853',
        rtol=1e-8,
        atol=1e-10
    )

    return sol




In [ ]:
sol = EvenSolver_single_basis(0.5, profile, bg_packed, r_tab)

r_vals = sol.t
X_vals = sol.y[0] + 1j*sol.y[1]
H1_vals = sol.y[4] + 1j*sol.y[5]
K_vals  = sol.y[6] + 1j*sol.y[7]

print("H1(R) =", H1_vals[-1])
print("K(R)  =", K_vals[-1])

In [ ]:
def Zerilli(omega, R, sol_surface, bg_tab, l=2):

    M   = bg_tab['m'][-1]
    nuR = bg_tab['nu'][-1]

    #XhR = sol_surface[0] + 1j * sol_surface[1]
    #WhR = sol_surface[2] + 1j * sol_surface[3]
    H1R = sol_surface[4] + 1j * sol_surface[5]
    KR  = sol_surface[6] + 1j * sol_surface[7]
    f = 1 - (2*M)/R

    lam = (l - 1) * (l + 2)
    D   = lam * R + 6.0 * M

    ZR = R**3 *((2*M-R)*H1R + R*KR)/(3*M + 2*R)  #(R**l * (2.0 * R**2 / D)   * (KR - cm.exp(nuR) * H1R)   )

    dZR = (R**2 *(6*(2*M**3 + M**2 *R + M*R**2 - R**3)*H1R + R*(-3*M**2 - 6*M*R + 2*R**2)*KR))/((2*M-R)*(3*M + 2*R)**2)   #(1/f) * ( R**l * (  ( 2.0 * lam * R**2 - 6.0 * M * (lam * R + 2.0 * M) ) * KR   + cm.exp(2.0 * nuR)  * (  lam * l * (l + 1) * R**2 + 6.0 * M * (lam * R + 4.0 * M)   ) * H1R ) / D**2   )
    #
    return ZR, dZR

def ZerilliR(omega, R, sol_surface, bg_tab):
    return Zerilli(omega, R, sol_surface, bg_tab, l=2)


def Chandrasekhar(om, Z, dZ, R, M, l=2):
    A = (l*(l+1)/2) - 1
    B = 6*M
    Kappa = 4*A*(A+1)
    f = 1 - (2*M)/R
    F = (R-2*M)/(2*R**2 *(A*R + 3*M))

    QR = ((Kappa + 2*B**2 *F)*Z - (2*B*f)*dZ)/(Kappa - 2j *B*om)

    dQR = (-2*B*(-om**2 + Kappa*F + B**2 *F**2)*Z + (Kappa + 2*B**2 *F)*f*dZ)/(Kappa - 2j *B*om)

    return QR, dQR


In [ ]:
Z, dZ = ZerilliR(1, R, EvenSolverNumbaZerilli(1, r_tab, bg_tab)[1], bg_tab=bg_tab)
print("Z = ", Z)
print("dZ = ", dZ)
print("Z'/Z at 1:", (2*M/(1-2*C))*dZ/Z, "for om = 1")
del Z, dZ

# Single basis shooting

In [ ]:

@njit
def even_rhs(x, y, x_grid, r_grid, bg_tab, src_tab, om, l=2):

    dydr = np.zeros_like(y)
    r = x #interp(x, x_grid, r_grid)
    #print("Interpolated r:", r, "for x =", x, "with grid from", r_grid[0], "to", r_grid[-1], "x_grid:", x_grid[0], "to", x_grid[-1])
    # unpack background
    m   = interp(r, r_grid, bg_tab[0])
    p   = interp(r, r_grid, bg_tab[1])
    eps = interp(r, r_grid, bg_tab[2])
    nu  = interp(r, r_grid, bg_tab[3])
    lam = interp(r, r_grid, bg_tab[4])
    lam_p = interp(r, r_grid, bg_tab[5])
    nu_p  = interp(r, r_grid, bg_tab[6])
    p_prime   = interp(r, r_grid, bg_tab[7])
    eps_prime = interp(r, r_grid, bg_tab[8])
    cs2   = interp(r, r_grid, bg_tab[9])

    # unpack sources (all zero for perfect fluid)
    S01  = interp(r, r_grid, src_tab[0])
    S01p = interp(r, r_grid, src_tab[1])
    S00  = interp(r, r_grid, src_tab[2])
    S0A  = interp(r, r_grid, src_tab[3])
    SZ   = interp(r, r_grid, src_tab[4])
    S0   = interp(r, r_grid, src_tab[5])
    S0p  = interp(r, r_grid, src_tab[6])
    S1   = interp(r, r_grid, src_tab[7])
    S1p  = interp(r, r_grid, src_tab[8])
    SOm  = interp(r, r_grid, src_tab[9])


    exp_lam = np.exp(lam)
    exp_nu  = np.exp(nu)

        # unpack state
    X  = y[0] + 1j*y[1]
    W  = y[2] + 1j*y[3]
    H1 = y[4] + 1j*y[5]
    K  = y[6] + 1j*y[7]

    Eh = X / cs2

    # ---- algebraic H0 and V here ----
    # (use your already corrected expressions)

    #Sighnum = cm.exp(-lam + (nu/2))*(2*cm.exp(lam)*r*om*S00*rho + p *(2*cm.exp(lam)*r*om*S00 + 1j*(12*cm.exp(lam)*S0A - 2*r* S01p + S01*(-4+r*lam_p - r*nu_p))) + 1j*(2*rho*(6*cm.exp(lam)*S0A - r*S01p) + S01*(2*r*n0*T*sp + rho*(-4+r*lam_p - r*nu_p))))
    Sigh = 0 #Sighnum/(2*r**3 *om*n0*T*(rho+p))
    drhods = 0

    Eh = (X/cs2) + drhods*Sigh


    Hdenom = r*(2*r + 3*m + 4*pi*r**3 *p) 
    H = (8*pi*r**4 * X * np.exp(-nu/2) + 2*r**2 *K - np.exp(-nu)*r**4 *om**2 *K + np.exp(lam)*r*K*m - 3*np.exp(lam)*K*m**2 + 4*np.exp(lam)*pi*r**4 *p*K - 16*np.exp(lam)*pi*r**3 *K *m*p - 16*np.exp(lam)*pi**2 *r**6 *K *p**2 + H1*(-3*r*m + r**4 *(np.exp(-lam-nu)*om**2 - 12*pi*p)) - 8*pi*r**2 *S0 - 16*np.exp(-lam)*pi*r**2 *S1)/Hdenom 

    Vhnum1 = np.exp(-lam + (nu/2))*(np.exp(lam) * r**2 * H *(p+eps) + 2*(-2*np.exp(lam)*SZ + np.exp(lam)*SOm + S1*(2+np.exp(lam)+4*np.exp(lam)*r**2 *(p-eps)) - np.exp(lam/2)*r*W*p_prime + r*S1p))
    Vhnum2 = np.exp(nu/2) *(2*X - (Vhnum1/r**2))
    Vh = Vhnum2/(2*om**2 *(eps+p))

    dH1dr = (np.exp(lam)/r) * H + (4*pi*r*(eps-p)*np.exp(lam) - (2*m*np.exp(lam)/r**2) - (3/r))*H1 + (np.exp(lam)/r)*K - 16*pi*((eps+p)/r)*np.exp(lam)*Vh + 16*pi*(np.exp(lam)/r) * SZ
    dKdr = (H/r) + (l*(l+1)/2*r) *H1 + ((nu_p/2) - ((l+1)/r))*K - 8*pi*((eps+p)/r)*np.exp(lam/2)*W + (16*pi/r)*SZ
    dWhdr = (r/2)*np.exp(lam/2)*H + r*np.exp(lam/2)*K + (r*np.exp((lam-nu)/2)/(eps+p)) * Eh - W*(l+1)/r -(l*(l+1)/r)*np.exp(lam/2)*Vh - np.exp(lam/2)*S00/(r*(eps+p)) + 8*pi*r*np.exp(lam/2)*SZ
    dXhdr = ((eps+p)/2)*np.exp(nu/2)*((1/r)-(nu_p/2))*H + ((eps+p)/2)*np.exp(nu/2)*(om**2 *r*np.exp(-nu) + (l*(l+1)/2*r))*H1 + ((eps+p)/2)*np.exp(nu/2)*((3*nu_p/2)-(1/r))*K - (l/r)*X - ((eps+p)/r)*np.exp((lam+nu)/2)*(om**2 *np.exp(-nu) + (np.exp(-lam)/4*r**2) *(np.exp(2*lam)*(1+8*pi*r**2 *p)**2 + 2*np.exp(lam)*(3 + 8*pi*r**2 *p - 7)) )*W + (l*(l+1)/r**2)*np.exp(nu/2)*p_prime*Vh + (np.exp(nu/2)/(2*r**3))*(2*l-1 +np.exp(lam)*(1+8*pi*r**2 *p))*S0 + (np.exp(nu/2)/r**2)*S0p - (np.exp(nu/2)/r**3)*(l*(l+1) + 8*pi*r**2 *(eps+p))*S1 - (2*np.exp(nu/2)/r**3)*SOm
    drdx = 1 #p / p_prime

    dH1dx = drdx * dH1dr
    dKdx  = drdx * dKdr
    dWhdx = drdx * dWhdr
    dXdx  = drdx * dXhdr

    print("At r =", r, "dXdx =", dXdx, "dWhdx =", dWhdx, "dH1dx =", dH1dx, "dKdx =", dKdx)
    return np.array([dXdx.real, dXdx.imag, dWhdx.real, dWhdx.imag, dH1dx.real, dH1dx.imag, dKdx.real, dKdx.imag])

In [ ]:
def init_conds_shooting(r0, omega, profile, A):

    p0   = profile['p'][0]
    eps0 = profile['eps'][0]
    nu0  = profile['nu'][0]
    pi = np.pi

    W0 = 1.0    
    K0 = A

    X0 = (1/6)*np.exp(-nu0/2)*(p0+eps0)*(
            -3*omega**2 *W0
            + np.exp(nu0)*(3*K0 + 8*pi*W0*(3*p0+eps0))
         ) 

    H10 = (2/3)*(K0 + 4*pi*W0)*(p0+eps0)

    return np.array([
        X0.real, X0.imag,
        W0.real, W0.imag,
        H10.real, H10.imag,
        K0.real, K0.imag
    ])


def integrate_for_A(A, omega, profile,
                    x_tab, r_tab, bg_packed, src_packed):
    
    start = 5   # avoid center singularity
    x0 = x_tab[start]#[-start]
    xf = x_tab[-1]

    y0 = init_conds_shooting(r_tab[start], omega, profile, A)
    print("Initial conditions for A =", A, ":", y0)
    sol = solve_ivp(
        lambda x,y: even_rhs(
            x,y,x_tab,r_tab,bg_packed,src_packed,omega
        ),
        (x0, xf),
        y0,
        method='Radau',
        rtol=1e-8,
        atol=1e-10
    )

    return sol

def X_surface_for_A(A, omega, profile,
                    x_tab, r_tab, bg_packed, src_packed):

    sol = integrate_for_A(A, omega, profile,
                          x_tab, r_tab, bg_packed, src_packed)

    X_R = sol.y[0,-1] + 1j*sol.y[1,-1]

    return X_R # usually X is real for real omega



def solve_shooting(omega, profile,
                   x_tab, r_tab, bg_packed, src_packed):

    def F(A):
        return X_surface_for_A(A, omega, profile,
                               x_tab, r_tab, bg_packed, src_packed)

    sol_root = root_scalar(F, bracket=[-10, 10], method='brentq')

    A_star = sol_root.root

    sol = integrate_for_A(A_star, omega, profile,
                          x_tab, r_tab, bg_packed, src_packed)

    return A_star, sol

In [ ]:
r_tab[5]

In [ ]:
# Remove p=0 surface point
mask = bg_packed[1] > 0.0

om = 1e-2 + 1e-2j

src_tab = make_source(SourceTensor(), r_tab, om)
src_packed = pack_source(src_tab)

p_tab = bg_packed[1][mask]
r_tab = r_tab[mask]
bg_packed = bg_packed[:, mask]
src_packed = src_packed[:, mask]

x_tab = r_tab #np.log(p_tab)

# IMPORTANT: sort so x is monotonic increasing
order = np.argsort(x_tab)

x_tab = x_tab[order]
r_tab = r_tab[order]
bg_packed = bg_packed[:, order]
src_packed = src_packed[:, order]

In [ ]:
print("x_tab from", x_tab[0], "to", x_tab[-1])
print("r_tab from", r_tab[0], "to", r_tab[-1])

In [ ]:
for A in [-5, -1, 0, 1, 5]:
    val = X_surface_for_A(A, om, profile,
                          x_tab, r_tab,
                          bg_packed, src_packed)
    print(A, val)